# Task 1 — Polyp Characterization with MobileNetV3

**Data:** CCE (colon capsule endoscopy) RGB images, 2 classes:
non-neoplasia (0) and neoplasia (1). Resized to 3x224x224 and normalized.

**Split:** by **patient id**, from `train_data.txt` / `valid_data.txt` / `test_data.txt`.
Not by image. Several frames can show the same polyp, so an image-level split would
leak the same lesion into train and test and inflate every number in the report.

**Model:** MobileNetV3-Large, fine-tuned from ImageNet weights.
**Early stopping:** patience 5, restores the best-validation weights.

## What to write about the model (report section 1)

- **Motivation.** MobileNet targets phones and embedded devices: high accuracy at a
  small fraction of the compute. That matters for CCE, where the long-term goal is
  screening at scale rather than one model on one GPU.
- **Key component — depthwise separable convolution.** A standard conv mixes space
  *and* channels in one operation. MobileNet splits it in two: a **depthwise** conv
  filters each input channel on its own, then a **pointwise** 1x1 conv mixes the
  channels. For a 3x3 kernel with C input and N output channels the cost drops from
  `9*C*N` to `9*C + C*N` — roughly 8-9x cheaper for the same receptive field.
- **V3 specifics.** Inverted residuals with linear bottlenecks (v2), plus
  squeeze-and-excitation blocks, the h-swish activation, and a architecture found by
  neural architecture search.
- **Limitations.** Fewer parameters means less capacity; on a small, noisy, low-contrast
  dataset like CCE that can cut both ways — less overfitting, but also a lower ceiling.
  Depthwise convs are also poorly served by some GPU kernels, so the wall-clock speedup
  is smaller than the FLOP count suggests.

In [ ]:
import os
# moved into notebooks/; walk up to the repo root so src/ and data/ resolve regardless of launch cwd
while not os.path.exists("requirements.txt") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from src.metrics import best_threshold, binary_metrics, plot_confusion, plot_roc
from src.polyp import build_model, evaluate, fit, load_polyp_splits, pos_weight_from
from src.results import results_table, save_run
from src.seed import get_device, set_seed

SEED = 27
set_seed(SEED)
device = get_device()
print("device:", device)

In [ ]:
CONFIG = {
    "model": "mobilenet_v3_large",
    "pretrained": True,        # False = train from scratch; say which in the report
    "image_size": 224,
    "batch_size": 32,
    "lr": 1e-4,
    "n_epochs": 50,
    "patience": 5,
    "augment": True,
    "seed": SEED,
}

train_loader, val_loader, test_loader = load_polyp_splits(
    root=".",
    image_size=CONFIG["image_size"],
    batch_size=CONFIG["batch_size"],
    augment=CONFIG["augment"],
)

In [ ]:
images, labels, names = next(iter(train_loader))
class_names = ["non-neoplasia", "neoplasia"]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.ravel(), range(min(8, len(images)))):
    ax.imshow((images[i].permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1))
    ax.set_title(f"{class_names[int(labels[i])]}\n{names[i]}", fontsize=7)
    ax.axis("off")
fig.tight_layout()

## Model and loss

`pos_weight` rescales the positive class inside `BCEWithLogitsLoss` so the rarer class
is not ignored. If your split turns out roughly balanced, drop it and say so.

In [ ]:
model = build_model(CONFIG["model"], pretrained=CONFIG["pretrained"]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_from(train_loader, device))
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

print(f"{CONFIG['model']}  pretrained={CONFIG['pretrained']}  "
      f"params {sum(p.numel() for p in model.parameters()):,}")
print("pos_weight:", criterion.pos_weight.item())

In [ ]:
history = fit(model, train_loader, val_loader, criterion, optimizer, device,
              n_epochs=CONFIG["n_epochs"], patience=CONFIG["patience"], scheduler=scheduler)

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(epochs, history["train_loss"], label="train")
ax1.plot(epochs, history["val_loss"], label="validation")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Loss"); ax1.legend()
ax2.plot(epochs, history["train_auc"], label="train")
ax2.plot(epochs, history["val_auc"], label="validation")
ax2.set(xlabel="Epoch", ylabel="AUC", title="AUC"); ax2.legend()
for ax in (ax1, ax2):
    ax.axvline(history["best_epoch"], color="grey", ls=":", lw=1)
fig.tight_layout()

## Decision threshold

The default 0.5 is arbitrary. Pick the threshold on the **validation** set (Youden's J
= the point maximising sensitivity + specificity - 1), then apply that fixed number to
test. Choosing it on test would be tuning on test.

For a screening tool you may deliberately prefer a lower threshold: missing a neoplastic
polyp (false negative) is far more costly than sending a healthy patient for a
colonoscopy (false positive). If you shift it, justify it in those terms.

In [ ]:
_, y_val, p_val, _ = evaluate(model, val_loader, criterion, device)
threshold = best_threshold(y_val, p_val)
print(f"threshold chosen on validation: {threshold:.3f}")

test_loss, y_test, p_test, test_names = evaluate(model, test_loader, criterion, device)
test_metrics = binary_metrics(y_test, p_test, threshold)

for name, value in test_metrics.items():
    print(f"{name:>12}: {value:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
plot_roc(y_test, p_test, label=CONFIG["model"], ax=ax1)
plot_confusion(y_test, (p_test >= threshold).astype(int), class_names=class_names, ax=ax2)
fig.tight_layout()
fig.savefig("roc_confusion_mobilenet.png", dpi=150)

In [ ]:
run_dir = save_run(
    tag=CONFIG["model"],
    config={**CONFIG, "threshold": threshold},
    metrics=test_metrics,
    history=history,
    model=model,
)

## Results table (report Table 1)

Fill in your teammates' numbers, then paste the printed markdown into the report.

In [ ]:
rows = {
    f"MobileNetV3 ({'pretrained' if CONFIG['pretrained'] else 'scratch'})": test_metrics,
    # "member2 model": {"accuracy": .., "sensitivity": .., "specificity": .., "f1": .., "auc": ..},
    # "member3 model": {...},
}
print(results_table(rows, ["accuracy", "sensitivity", "specificity", "f1", "auc"]))

## Discussion — what to actually argue

- **Sensitivity vs specificity.** Report both, not just accuracy. On an unbalanced set
  a model that always predicts "non-neoplasia" can look accurate and be useless.
- **Why one model beat another.** Parameter count is not the answer on its own. Compare
  pretrained vs scratch first — on a dataset this small that usually dominates the
  architecture choice.
- **Patient-level split.** Point out that your numbers are patient-level. If a teammate
  split by image, their scores are not comparable to yours and you should say why.
- **Limitations.** Low-resolution CCE frames, motion blur, partial views, few patients.
  Note how many *patients* (not images) are in test — with a handful of patients the
  confidence interval on AUC is wide.
- **Future work.** More augmentation, test-time augmentation, cross-validation over
  patient folds instead of a single split.